# Hakisa 2.0 — GPU training on Colab

This notebook runs the parts of Hakisa that benefit from a big GPU:

1. **Study (BC)** on your recordings (uploaded to Google Drive) or on DanmakuDojo demos
2. **System 2 with the real pre-trained `Qwen/Qwen3-0.6B`** (LoRA) on top of a trained System 1
3. Latency benchmark on the GPU

Real-time play and RL on Jigoku Kisetsukan / Bullet Heaven 2 / LoL must run on your Windows PC
(the game has to be on the same machine as Hakisa). Recording also happens on the PC.

Runtime → Change runtime type → GPU.

In [ ]:
# 1. Get the code (branch claude/hakisa-revival-mpe21d until it is merged)
!git clone -b claude/hakisa-revival-mpe21d https://github.com/Martyn0324/Hakisa.git
%cd Hakisa
!pip install -q -e .[llm,dev,viz]
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [ ]:
# 2. (Optional) persist data and checkpoints on Google Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/hakisa'
!mkdir -p $DRIVE/data $DRIVE/runs

In [ ]:
# 3. Sanity: tests + latency of every game network on this GPU (fp16)
!python -m pytest -q -x
!python scripts/benchmark_latency.py --device cuda --half

## Study on your own recordings

On your PC: `hakisa record --game jigoku --out data/jigoku` (see `docs/games/jigoku.md`), then upload
the `data/jigoku` folder to `MyDrive/hakisa/data/jigoku`.

In [ ]:
GAME = 'jigoku'          # jigoku | bullet_heaven2 | lol
!hakisa train-bc --game $GAME --data $DRIVE/data/$GAME --out $DRIVE/runs/$GAME/bc --device cuda --workers 2 --batch-size 128
# Then copy runs/<game>/bc/best.pt back to your PC and run:  hakisa play --game <game> --ckpt best.pt

## DanmakuDojo: BC -> RL experiment at a larger scale (no recordings needed)

In [ ]:
!python scripts/dojo_experiments.py --scheme keyboard --demo-steps 300000 --bc-epochs 8 --rl-steps 3000000 --threads 8 \
    --data $DRIVE/data/dojo_keyboard --runs $DRIVE/runs/dojo_keyboard --out $DRIVE/results/dojo_keyboard

## System 2 with the real Qwen3-0.6B

Needs a trained keyboard System 1 (`runs/dojo_keyboard/bc/best.pt`, from the cell above or the repo
experiments). The script fine-tunes an intent-conditioned System 1, trains the planner (LoRA on Qwen3-0.6B
+ a projector from Hakisa's own visual tokens) on hindsight plans, evaluates System 1 alone vs System 1 +
System 2, runs GRPO, and evaluates again.

In [ ]:
!python scripts/s2_experiment.py --llm Qwen/Qwen3-0.6B --device cuda --train-mode lora \
    --data $DRIVE/data/dojo_keyboard --bc $DRIVE/runs/dojo_keyboard/bc/best.pt \
    --runs $DRIVE/runs/dojo_keyboard --out $DRIVE/results/system2_qwen3 --threads 8 --grpo-steps 300

In [ ]:
# The planner directory (runs/dojo_keyboard/s2_grpo) holds only the projector + LoRA weights (a few MB);
# the base Qwen3-0.6B is downloaded from the Hub when loading. On the PC (4 GB GPU):
#   hakisa play --game <game> --ckpt <system1.pt> --planner <s2 dir>
!du -sh $DRIVE/runs/dojo_keyboard/s2_grpo